In [1]:
from pathlib import Path
import sys
import os
import random

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm import tqdm

project_root = Path.cwd()
if project_root.name == "embedding_pipeline":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from embedding_pipeline.stft import stft

In [2]:
class AudioFileDataset(Dataset):
    def __init__(self, samples, max_time_windows=256):
        self.samples = samples
        self.max_time_windows = max_time_windows

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        stft_tensor = process_clip(file_path, self.max_time_windows)
        target = torch.tensor([label], dtype=torch.float32)
        return stft_tensor, target
    

def process_clip(file_path, max_time_windows=256):
    magnitude_db, _ = stft(file_path)
    stft_tensor = torch.tensor(magnitude_db).unsqueeze(0).float()  # [1, 512, W]
    width = stft_tensor.shape[-1]

    # clip long samples and pad short samples so all tensors have the same width
    if width > max_time_windows:
        stft_tensor = stft_tensor[:, :, :max_time_windows]
    elif width < max_time_windows:
        pad_width = max_time_windows - width
        stft_tensor = torch.nn.functional.pad(stft_tensor, (0, pad_width))

    return stft_tensor


def collect_labeled_files(root_dir, label, max_count):
    samples = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            samples.append((os.path.join(root, file), label))
            if len(samples) >= max_count:
                return samples
    return samples

In [3]:
max_files = 1000
max_time_windows = 256
batch_size = 32
class_names = {"fake": 1, "real": 0}


fake_samples = collect_labeled_files(
    "../data/release_in_the_wild/train/fake", label=1, max_count=max_files
)
real_samples = collect_labeled_files(
    "../data/release_in_the_wild/train/real", label=0, max_count=max_files
)

dataset = fake_samples + real_samples
random.shuffle(dataset)
print("dataset created with", len(dataset), "samples")


# split into train/test file lists
train_to_test_ratio = 0.8
train_size = int(len(dataset) * train_to_test_ratio)

train_samples = dataset[:train_size]
test_samples = dataset[train_size:]

train_dataset = AudioFileDataset(train_samples, max_time_windows)
test_dataset = AudioFileDataset(test_samples, max_time_windows)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

dataset created with 2000 samples


In [4]:
from embedding_pipeline.embedder import (
    Embedder,
    ContrastiveLoss
)

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_dim = 128

model = Embedder(embedding_dim=embedding_dim).to(device)

contrastive_loss_fn = ContrastiveLoss(temperature=0.07).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"Model initialized with embedding_dim={embedding_dim}")

Model initialized with embedding_dim=128


In [5]:
epochs = 10

for epoch in range(epochs):
    # training phase
    model.train()
    epoch_loss = 0.0
    
    for batch_idx, (data, target) in enumerate(train_dataloader):
        data = data.to(device)
        batch_target = target.to(device)
        
        # forward pass
        embeddings = model(data) # [batch_size*Hf*Wf, embedding_dim]

        # compute loss
        loss = contrastive_loss_fn(embeddings, batch_target)
        
        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if batch_idx % 2 == 0:
            print(f"batch progress: epoch {epoch+1} {(100 *(batch_idx+1)/len(train_dataloader)):.2f}% loss: {loss.item():.6f}")
    
    # validation phase
    model.eval()
    test_loss = 0.0
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_dataloader):
            data = data.to(device)
            batch_target = target.to(device)
            embeddings = model(data)
            loss = contrastive_loss_fn(embeddings, batch_target)
            test_loss += loss.item()
    
    total_test_loss = test_loss / len(test_dataloader)
    print('epoch: {}, test loss: {:.6f}'.format(
        epoch + 1,
        total_test_loss,
        ))

    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch + 1,
        "batch": batch_idx,
    }, "embedder_checkpoint.pth")
    print('model checkpoint saved')


c:\Users\whyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


batch progress: epoch 1 2.00% loss: 11.592199
batch progress: epoch 1 6.00% loss: 11.123501
batch progress: epoch 1 10.00% loss: 10.808168
batch progress: epoch 1 14.00% loss: 10.550817
batch progress: epoch 1 18.00% loss: 10.362396
batch progress: epoch 1 22.00% loss: 10.194770
batch progress: epoch 1 26.00% loss: 10.094388
batch progress: epoch 1 30.00% loss: 10.000994
batch progress: epoch 1 34.00% loss: 9.951733
batch progress: epoch 1 38.00% loss: 9.904564
batch progress: epoch 1 42.00% loss: 9.869759
batch progress: epoch 1 46.00% loss: 9.837269
batch progress: epoch 1 50.00% loss: 9.819214
batch progress: epoch 1 54.00% loss: 9.809457
batch progress: epoch 1 58.00% loss: 9.791691
batch progress: epoch 1 62.00% loss: 9.780567
batch progress: epoch 1 66.00% loss: 9.772921
batch progress: epoch 1 70.00% loss: 9.764729
batch progress: epoch 1 74.00% loss: 9.760970
batch progress: epoch 1 78.00% loss: 9.756396
batch progress: epoch 1 82.00% loss: 9.752529
batch progress: epoch 1 86.0